# Depth Anything V2 Metric Depth - Interactive Pixel Viewer


It also includes a video workflow that runs inference frame by frame and builds a Plotly slider for reviewing depth predictions across frames.
This notebook runs the same metric-depth inference you tested with `models/depth/Depth-Anything-V2/metric_depth/run.py` and saves the same raw metric depth array as `*_raw_depth_meter.npy`.

The raw `.npy` output is the model's depth estimate in meters. Use the Plotly figures below to hover over pixels and inspect their `x`, `y`, and depth values.

## Dependency Note

If Plotly is not installed in the active notebook kernel, run this once in a new cell and then restart the kernel:

```python
%pip install plotly
```

In [ ]:
1

In [ ]:
from pathlib import Path
import sys

# The notebook is intended to live at the project root. The fallback keeps it
# working if Jupyter starts in a different current working directory.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "models" / "depth" / "Depth-Anything-V2" / "metric_depth").exists():
    PROJECT_ROOT = Path(r"C:\Users\jjaramil\OneDrive - InterSystems Corporation\Documents\Muay-ThAI")

DEPTH_REPO = PROJECT_ROOT / "models" / "depth" / "Depth-Anything-V2" / "metric_depth"
IMG_PATH = PROJECT_ROOT / "media" / "images" / "teep.jpg"
VIDEO_PATH = PROJECT_ROOT / "media" / "videos" / "Rodtang-taetat-2.mp4"
OUTDIR = PROJECT_ROOT / "output"
CAPTURE_RUN_DIR = OUTDIR / "rodtang-taetat-2__depth-anything-v2-metric-hypersim-vitl__20260707-142906"
CHECKPOINT_PATH = DEPTH_REPO / "checkpoints" / "depth_anything_v2_metric_hypersim_vitl.pth"

# These match the command you successfully tested.
ENCODER = "vitl"
MAX_DEPTH = 20.0
INPUT_SIZE = 200
SAVE_NUMPY = True

# Video settings. The slider covers every processed frame; set VIDEO_MAX_FRAMES = None for the whole video.
VIDEO_START_FRAME = 0
VIDEO_FRAME_STRIDE = 1
VIDEO_MAX_FRAMES = 60
VIDEO_PLOT_STRIDE = 2
VIDEO_SURFACE_STRIDE = 8
VIDEO_SAVE_NUMPY = True
VIDEO_SAVE_PREVIEW_MP4 = True

# capture_depth.py output settings. These cells load saved artifacts instead of running inference again.
CAPTURE_MAX_FRAMES = None
CAPTURE_PLOT_STRIDE = 2
CAPTURE_SURFACE_STRIDE = 8

assert DEPTH_REPO.exists(), f"Metric depth repo not found: {DEPTH_REPO}"
assert IMG_PATH.exists(), f"Input image not found: {IMG_PATH}"
assert CHECKPOINT_PATH.exists(), f"Checkpoint not found: {CHECKPOINT_PATH}"
assert VIDEO_PATH.exists(), f"Video file not found: {VIDEO_PATH}"
assert CAPTURE_RUN_DIR.exists(), f"Capture run not found: {CAPTURE_RUN_DIR}"

if str(DEPTH_REPO) not in sys.path:
    sys.path.insert(0, str(DEPTH_REPO))

OUTDIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Image:        {IMG_PATH}")
print(f"Video:       {VIDEO_PATH}")
print(f"Capture run: {CAPTURE_RUN_DIR}")
print(f"Checkpoint:   {CHECKPOINT_PATH}")
print(f"Output dir:   {OUTDIR}")

In [ ]:
import cv2
import matplotlib
import numpy as np
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots

from depth_anything_v2.dpt import DepthAnythingV2

if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

MODEL_CONFIGS = {
    "vits": {"encoder": "vits", "features": 64, "out_channels": [48, 96, 192, 384]},
    "vitb": {"encoder": "vitb", "features": 128, "out_channels": [96, 192, 384, 768]},
    "vitl": {"encoder": "vitl", "features": 256, "out_channels": [256, 512, 1024, 1024]},
    "vitg": {"encoder": "vitg", "features": 384, "out_channels": [1536, 1536, 1536, 1536]},
}

depth_model = DepthAnythingV2(**{**MODEL_CONFIGS[ENCODER], "max_depth": MAX_DEPTH})

try:
    state_dict = torch.load(str(CHECKPOINT_PATH), map_location="cpu", weights_only=True)
except TypeError:
    state_dict = torch.load(str(CHECKPOINT_PATH), map_location="cpu")

depth_model.load_state_dict(state_dict)
depth_model = depth_model.to(DEVICE).eval()

print(f"Loaded Depth Anything V2 metric model: encoder={ENCODER}, max_depth={MAX_DEPTH}, device={DEVICE}")

In [ ]:
raw_bgr = cv2.imread(str(IMG_PATH))
if raw_bgr is None:
    raise ValueError(f"OpenCV could not read image: {IMG_PATH}")

raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)

with torch.inference_mode():
    depth_m = depth_model.infer_image(raw_bgr, INPUT_SIZE)

depth_m = np.asarray(depth_m, dtype=np.float32)
finite_depth = depth_m[np.isfinite(depth_m)]
if finite_depth.size == 0:
    raise ValueError("The model returned no finite depth values.")

stem = IMG_PATH.stem
raw_depth_path = OUTDIR / f"{stem}_raw_depth_meter.npy"
vis_path = OUTDIR / f"{stem}.png"

if SAVE_NUMPY:
    np.save(raw_depth_path, depth_m)

# Match metric_depth/run.py: normalize only for the PNG visualization.
depth_min = float(finite_depth.min())
depth_max = float(finite_depth.max())
scale = max(depth_max - depth_min, np.finfo(np.float32).eps)
depth_uint8 = ((depth_m - depth_min) / scale * 255.0).clip(0, 255).astype(np.uint8)

cmap = matplotlib.colormaps.get_cmap("Spectral")
depth_color_bgr = (cmap(depth_uint8)[:, :, :3] * 255)[:, :, ::-1].astype(np.uint8)
split_region = np.ones((raw_bgr.shape[0], 50, 3), dtype=np.uint8) * 255
combined_result = cv2.hconcat([raw_bgr, split_region, depth_color_bgr])
cv2.imwrite(str(vis_path), combined_result)

print(f"Depth array shape: {depth_m.shape[0]} rows x {depth_m.shape[1]} columns")
print(f"Depth range:       {depth_min:.4f} m to {depth_max:.4f} m")
print(f"Depth mean:        {float(finite_depth.mean()):.4f} m")
print(f"Saved raw depth:   {raw_depth_path}")
print(f"Saved PNG view:    {vis_path}")

## Interactive Metric Depth Heatmap

Hover over the heatmap to inspect the model's metric depth estimate for each pixel.

In [ ]:
height, width = depth_m.shape
plot_width = 900
plot_height = min(900, max(500, int(plot_width * height / max(width, 1))))

fig_depth = go.Figure(
    data=go.Heatmap(
        z=depth_m,
        x=np.arange(width),
        y=np.arange(height),
        colorscale="Spectral",
        colorbar={"title": "Depth (m)"},
        hovertemplate="x: %{x}<br>y: %{y}<br>depth: %{z:.3f} m<extra></extra>",
    )
)

fig_depth.update_layout(
    title="Metric depth map",
    width=plot_width,
    height=plot_height,
    margin={"l": 60, "r": 30, "t": 55, "b": 55},
)
fig_depth.update_xaxes(title="x pixel", constrain="domain")
fig_depth.update_yaxes(title="y pixel", autorange="reversed", scaleanchor="x", scaleratio=1)
fig_depth.show()

## Original Image With Depth Hover

This keeps the RGB image visible while reporting the metric depth value for the hovered pixel.

In [ ]:
fig_rgb = go.Figure(
    data=go.Image(
        z=raw_rgb,
        customdata=depth_m,
        hovertemplate="x: %{x}<br>y: %{y}<br>depth: %{customdata:.3f} m<extra></extra>",
    )
)

fig_rgb.update_layout(
    title="Input image with metric depth hover",
    width=plot_width,
    height=plot_height,
    margin={"l": 60, "r": 30, "t": 55, "b": 55},
)
fig_rgb.update_xaxes(title="x pixel", showgrid=False, constrain="domain")
fig_rgb.update_yaxes(title="y pixel", showgrid=False, autorange="reversed", scaleanchor="x", scaleratio=1)
fig_rgb.show()

## Optional 3D Surface View

The 3D view is downsampled by default to keep the notebook responsive. Set `SURFACE_STRIDE = 1` if you want every pixel and your machine can handle it.

In [ ]:
SURFACE_STRIDE = max(1, min(depth_m.shape) // 250)

surface_depth = depth_m[::SURFACE_STRIDE, ::SURFACE_STRIDE]
surface_x = np.arange(0, width, SURFACE_STRIDE)
surface_y = np.arange(0, height, SURFACE_STRIDE)

fig_surface = go.Figure(
    data=go.Surface(
        z=surface_depth,
        x=surface_x,
        y=surface_y,
        colorscale="Viridis",
        colorbar={"title": "Depth (m)"},
        hovertemplate="x: %{x}<br>y: %{y}<br>depth: %{z:.3f} m<extra></extra>",
    )
)

fig_surface.update_layout(
    title=f"3D metric depth surface, stride={SURFACE_STRIDE}",
    width=900,
    height=700,
    margin={"l": 0, "r": 0, "t": 45, "b": 0},
    scene={
        "xaxis_title": "x pixel",
        "yaxis_title": "y pixel",
        "zaxis_title": "Depth (m)",
        "yaxis": {"autorange": "reversed"},
        "aspectmode": "manual",
        "aspectratio": {"x": width / max(height, 1), "y": 1, "z": 0.5},
    },
)
fig_surface.show()

## Exact Pixel Lookup

Use this helper when you want to query a specific pixel programmatically.

In [ ]:
def depth_at(x, y):
    """Return the metric depth in meters at pixel coordinate (x, y)."""
    x = int(x)
    y = int(y)
    if not (0 <= x < width and 0 <= y < height):
        raise IndexError(f"Pixel ({x}, {y}) is outside image bounds {width}x{height}")
    return float(depth_m[y, x])

center_x = width // 2
center_y = height // 2
print(f"Depth at center pixel ({center_x}, {center_y}): {depth_at(center_x, center_y):.4f} m")

## Load `capture_depth.py` Output

This section reads `CAPTURE_RUN_DIR` instead of running Depth Anything inference again. It loads the saved metadata, frame summaries, source video frames, and colorized depth video. If the run was created with `--save-frame-depths`, it also loads the raw metric depth arrays for per-pixel heatmaps, exact lookup, and 3D surfaces.


In [ ]:
import json
from pathlib import Path

import cv2
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

CAPTURE_RUN_DIR = Path(CAPTURE_RUN_DIR)
CAPTURE_PLOT_STRIDE = max(1, int(CAPTURE_PLOT_STRIDE))
CAPTURE_SURFACE_STRIDE = max(1, int(CAPTURE_SURFACE_STRIDE))

def resolve_capture_path(value):
    if value is None:
        return None
    candidate = Path(value)
    if candidate.is_absolute():
        return candidate
    return CAPTURE_RUN_DIR / candidate

metadata_files = sorted(CAPTURE_RUN_DIR.glob("*_metadata.json"))
if not metadata_files:
    raise FileNotFoundError(f"No *_metadata.json file found in {CAPTURE_RUN_DIR}")

capture_metadata_path = metadata_files[0]
capture_metadata = json.loads(capture_metadata_path.read_text(encoding="utf-8"))
frame_predictions_path = resolve_capture_path(capture_metadata.get("frame_predictions"))
depth_video_path = resolve_capture_path(capture_metadata.get("depth_video"))
source_video_path = resolve_capture_path(capture_metadata.get("source"))

if frame_predictions_path is None or not frame_predictions_path.exists():
    raise FileNotFoundError(f"Frame predictions JSONL not found: {frame_predictions_path}")
if depth_video_path is None or not depth_video_path.exists():
    raise FileNotFoundError(f"Depth video not found: {depth_video_path}")
if source_video_path is None or not source_video_path.exists():
    raise FileNotFoundError(f"Source video not found: {source_video_path}")

with frame_predictions_path.open("r", encoding="utf-8") as f:
    frame_predictions = [json.loads(line) for line in f if line.strip()]

if CAPTURE_MAX_FRAMES is not None:
    frame_predictions = frame_predictions[: int(CAPTURE_MAX_FRAMES)]
if not frame_predictions:
    raise ValueError("No frame predictions were loaded from the capture output.")

video_frame_numbers = [int(row["source_frame_index"]) for row in frame_predictions]
video_frame_times_s = [row.get("time_s", None) for row in frame_predictions]
video_frame_times_s = [np.nan if value is None else float(value) for value in video_frame_times_s]
capture_output_fps = float(capture_metadata.get("output_fps") or capture_metadata.get("source_fps") or 30.0)
preview_fps = capture_output_fps

def downsample_image(image_rgb):
    return image_rgb[::CAPTURE_PLOT_STRIDE, ::CAPTURE_PLOT_STRIDE]

def read_source_rgb_frames(video_path, source_frame_indices):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"OpenCV could not open source video: {video_path}")
    frames = []
    try:
        for frame_index in source_frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
            ok, frame_bgr = cap.read()
            if not ok:
                raise ValueError(f"Could not read source frame {frame_index} from {video_path}")
            frames.append(downsample_image(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)))
    finally:
        cap.release()
    return frames

def read_preview_rgb_frames(video_path, count):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"OpenCV could not open depth video: {video_path}")
    frames = []
    try:
        for processed_index in range(count):
            ok, frame_bgr = cap.read()
            if not ok:
                raise ValueError(f"Could not read depth preview frame {processed_index} from {video_path}")
            frames.append(downsample_image(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)))
    finally:
        cap.release()
    return frames

def load_depth_array(depth_path):
    loaded = np.load(depth_path)
    if isinstance(loaded, np.lib.npyio.NpzFile):
        try:
            if "depth" not in loaded.files:
                raise KeyError(f"Depth file does not contain a depth array: {depth_path}")
            depth = np.asarray(loaded["depth"], dtype=np.float32)
        finally:
            loaded.close()
        return depth
    return np.asarray(loaded, dtype=np.float32)

video_frames_rgb = read_source_rgb_frames(source_video_path, video_frame_numbers)
video_depth_preview_rgb = read_preview_rgb_frames(depth_video_path, len(frame_predictions))

candidate_depth_files = [resolve_capture_path(row.get("depth_file")) for row in frame_predictions]
video_depth_files = [path for path in candidate_depth_files if path is not None]
video_has_raw_depths = (
    len(video_depth_files) == len(frame_predictions)
    and all(path.exists() for path in video_depth_files)
)

if video_has_raw_depths:
    video_depths_m = [load_depth_array(path)[::CAPTURE_PLOT_STRIDE, ::CAPTURE_PLOT_STRIDE] for path in video_depth_files]
    video_depth_y = np.arange(0, load_depth_array(video_depth_files[0]).shape[0], CAPTURE_PLOT_STRIDE)[: video_depths_m[0].shape[0]]
    video_depth_x = np.arange(0, load_depth_array(video_depth_files[0]).shape[1], CAPTURE_PLOT_STRIDE)[: video_depths_m[0].shape[1]]
else:
    video_depths_m = []
    video_depth_y = None
    video_depth_x = None

plot_height, plot_width = video_frames_rgb[0].shape[:2]
video_plot_x = np.arange(0, int(capture_metadata.get("source_width") or plot_width * CAPTURE_PLOT_STRIDE), CAPTURE_PLOT_STRIDE)[:plot_width]
video_plot_y = np.arange(0, int(capture_metadata.get("source_height") or plot_height * CAPTURE_PLOT_STRIDE), CAPTURE_PLOT_STRIDE)[:plot_height]

capture_rerun_with_depths_command = (
    f'python capture_depth.py --source "{capture_metadata["source"]}" '
    f'--encoder {capture_metadata.get("encoder", "vitl")} '
    f'--max-depth {capture_metadata.get("max_depth", 20.0)} '
    f'--input-size {capture_metadata.get("input_size", 200)} '
    '--save-frame-depths'
)

def capture_frame_label(i):
    time_s = video_frame_times_s[i]
    if np.isfinite(time_s):
        return f"frame {video_frame_numbers[i]} ({time_s:.2f}s)"
    return f"frame {video_frame_numbers[i]}"

def capture_frame_stats(i):
    row = frame_predictions[i]
    return (
        f"min {row['depth_min_m']:.3f} m | "
        f"mean {row['depth_mean_m']:.3f} m | "
        f"max {row['depth_max_m']:.3f} m"
    )

print(f"Loaded capture run:      {CAPTURE_RUN_DIR}")
print(f"Metadata:                {capture_metadata_path}")
print(f"Frame summaries:         {frame_predictions_path}")
print(f"Source video:            {source_video_path}")
print(f"Colorized depth video:   {depth_video_path}")
print(f"Loaded frames:           {len(frame_predictions)}")
print(f"Display stride:          {CAPTURE_PLOT_STRIDE}")
print(f"""Raw metric frame depths: {"available" if video_has_raw_depths else "not saved in this run"}""")
if not video_has_raw_depths:
    print(f"Enable raw metric plots: {capture_rerun_with_depths_command}")


## Capture Output Slider

Use the slider to move through the frames loaded from `capture_depth.py`. If raw depth arrays are available, the right panel is a metric heatmap with per-pixel hover values; otherwise it shows the saved colorized depth video and frame-level metric summaries.


In [ ]:
def capture_hover_text(i):
    return capture_frame_stats(i).replace(" | ", "<br>")

if video_has_raw_depths:
    video_plot_depth_min = min(float(np.nanmin(depth)) for depth in video_depths_m)
    video_plot_depth_max = max(float(np.nanmax(depth)) for depth in video_depths_m)
    right_title = "Metric depth"
else:
    video_plot_depth_min = None
    video_plot_depth_max = None
    right_title = "Colorized depth video"

fig_video = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("RGB source frame", right_title),
    horizontal_spacing=0.06,
)

fig_video.add_trace(
    go.Image(
        z=video_frames_rgb[0],
        x0=0,
        y0=0,
        dx=CAPTURE_PLOT_STRIDE,
        dy=CAPTURE_PLOT_STRIDE,
        hovertemplate="x: %{x}<br>y: %{y}<br>" + capture_hover_text(0) + "<extra></extra>",
    ),
    row=1,
    col=1,
)

if video_has_raw_depths:
    fig_video.add_trace(
        go.Heatmap(
            z=video_depths_m[0],
            x=video_depth_x,
            y=video_depth_y,
            colorscale="Spectral",
            zmin=video_plot_depth_min,
            zmax=video_plot_depth_max,
            colorbar={"title": "Depth (m)"},
            hovertemplate="x: %{x}<br>y: %{y}<br>depth: %{z:.3f} m<extra></extra>",
        ),
        row=1,
        col=2,
    )
else:
    fig_video.add_trace(
        go.Image(
            z=video_depth_preview_rgb[0],
            x0=0,
            y0=0,
            dx=CAPTURE_PLOT_STRIDE,
            dy=CAPTURE_PLOT_STRIDE,
            hovertemplate="x: %{x}<br>y: %{y}<br>colorized preview<br>" + capture_hover_text(0) + "<extra></extra>",
        ),
        row=1,
        col=2,
    )

frame_data = []
for i in range(len(frame_predictions)):
    left = go.Image(
        z=video_frames_rgb[i],
        x0=0,
        y0=0,
        dx=CAPTURE_PLOT_STRIDE,
        dy=CAPTURE_PLOT_STRIDE,
        hovertemplate="x: %{x}<br>y: %{y}<br>" + capture_hover_text(i) + "<extra></extra>",
    )
    if video_has_raw_depths:
        right = go.Heatmap(
            z=video_depths_m[i],
            x=video_depth_x,
            y=video_depth_y,
            colorscale="Spectral",
            zmin=video_plot_depth_min,
            zmax=video_plot_depth_max,
            colorbar={"title": "Depth (m)"},
            hovertemplate="x: %{x}<br>y: %{y}<br>depth: %{z:.3f} m<extra></extra>",
        )
    else:
        right = go.Image(
            z=video_depth_preview_rgb[i],
            x0=0,
            y0=0,
            dx=CAPTURE_PLOT_STRIDE,
            dy=CAPTURE_PLOT_STRIDE,
            hovertemplate="x: %{x}<br>y: %{y}<br>colorized preview<br>" + capture_hover_text(i) + "<extra></extra>",
        )
    frame_data.append(
        go.Frame(
            name=str(i),
            traces=[0, 1],
            data=[left, right],
            layout=go.Layout(title_text=f"capture_depth.py output - {capture_frame_label(i)} | {capture_frame_stats(i)}"),
        )
    )

fig_video.frames = frame_data
slider_steps = [
    {
        "method": "animate",
        "label": str(video_frame_numbers[i]),
        "args": [
            [str(i)],
            {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}},
        ],
    }
    for i in range(len(frame_predictions))
]

play_duration_ms = max(20, int(1000 / max(capture_output_fps, 1)))
fig_video.update_layout(
    title=f"capture_depth.py output - {capture_frame_label(0)} | {capture_frame_stats(0)}",
    width=1150,
    height=650,
    margin={"l": 55, "r": 35, "t": 70, "b": 100},
    updatemenus=[
        {
            "type": "buttons",
            "direction": "left",
            "x": 0.0,
            "y": -0.12,
            "buttons": [
                {"label": "Play", "method": "animate", "args": [None, {"fromcurrent": True, "frame": {"duration": play_duration_ms, "redraw": True}, "transition": {"duration": 0}}]},
                {"label": "Pause", "method": "animate", "args": [[None], {"mode": "immediate", "frame": {"duration": 0, "redraw": False}, "transition": {"duration": 0}}]},
            ],
        }
    ],
    sliders=[{"active": 0, "currentvalue": {"prefix": "Source frame "}, "pad": {"t": 35}, "steps": slider_steps}],
)

fig_video.update_xaxes(title_text="x pixel", showgrid=False, row=1, col=1)
fig_video.update_yaxes(title_text="y pixel", showgrid=False, autorange="reversed", scaleanchor="x", scaleratio=1, row=1, col=1)
fig_video.update_xaxes(title_text="x pixel", showgrid=False, row=1, col=2)
fig_video.update_yaxes(title_text="y pixel", showgrid=False, autorange="reversed", scaleanchor="x2", scaleratio=1, row=1, col=2)
fig_video.show()


## Capture Output 3D Depth Surface Slider

This plot uses raw per-frame metric depth arrays saved by `capture_depth.py --save-frame-depths`. The named run currently has only frame summaries and a colorized depth video unless `depth_file` paths are present in the JSONL.


In [ ]:
if not video_has_raw_depths:
    raise ValueError(
        "This capture_depth.py run does not include raw per-frame depth arrays. "
        f"Run this first: {capture_rerun_with_depths_command}. Then point CAPTURE_RUN_DIR "
        "to that new output folder for metric heatmaps, exact pixel lookup, and 3D surfaces."
    )

surface_depths_m = [load_depth_array(path)[::CAPTURE_SURFACE_STRIDE, ::CAPTURE_SURFACE_STRIDE] for path in video_depth_files]
surface_height, surface_width = surface_depths_m[0].shape
full_frame_height, full_frame_width = load_depth_array(video_depth_files[0]).shape
surface_x = np.arange(0, full_frame_width, CAPTURE_SURFACE_STRIDE)[:surface_width]
surface_y = np.arange(0, full_frame_height, CAPTURE_SURFACE_STRIDE)[:surface_height]
surface_depth_min = min(float(np.nanmin(depth)) for depth in surface_depths_m)
surface_depth_max = max(float(np.nanmax(depth)) for depth in surface_depths_m)

fig_video_3d = go.Figure(
    data=[
        go.Surface(
            z=surface_depths_m[0],
            x=surface_x,
            y=surface_y,
            colorscale="Viridis",
            cmin=surface_depth_min,
            cmax=surface_depth_max,
            colorbar={"title": "Depth (m)"},
            hovertemplate="x: %{x}<br>y: %{y}<br>depth: %{z:.3f} m<extra></extra>",
        )
    ]
)

fig_video_3d.frames = [
    go.Frame(
        name=str(i),
        data=[
            go.Surface(
                z=surface_depths_m[i],
                x=surface_x,
                y=surface_y,
                colorscale="Viridis",
                cmin=surface_depth_min,
                cmax=surface_depth_max,
                colorbar={"title": "Depth (m)"},
                hovertemplate="x: %{x}<br>y: %{y}<br>depth: %{z:.3f} m<extra></extra>",
            )
        ],
        layout=go.Layout(title_text=f"capture_depth.py 3D depth - {capture_frame_label(i)} | {capture_frame_stats(i)}"),
    )
    for i in range(len(surface_depths_m))
]

surface_slider_steps = [
    {
        "method": "animate",
        "label": str(video_frame_numbers[i]),
        "args": [[str(i)], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}],
    }
    for i in range(len(surface_depths_m))
]

surface_play_duration_ms = max(20, int(1000 / max(capture_output_fps, 1)))
fig_video_3d.update_layout(
    title=f"capture_depth.py 3D depth - {capture_frame_label(0)} | {capture_frame_stats(0)}",
    width=1000,
    height=760,
    margin={"l": 0, "r": 0, "t": 55, "b": 90},
    scene={
        "xaxis_title": "x pixel",
        "yaxis_title": "y pixel",
        "zaxis_title": "Depth (m)",
        "xaxis": {"range": [0, full_frame_width - 1]},
        "yaxis": {"range": [full_frame_height - 1, 0]},
        "zaxis": {"range": [surface_depth_min, surface_depth_max]},
        "aspectmode": "manual",
        "aspectratio": {"x": full_frame_width / max(full_frame_height, 1), "y": 1, "z": 0.45},
        "camera": {"eye": {"x": 1.25, "y": -1.65, "z": 1.05}},
    },
    updatemenus=[
        {
            "type": "buttons",
            "direction": "left",
            "x": 0.0,
            "y": -0.08,
            "buttons": [
                {"label": "Play", "method": "animate", "args": [None, {"fromcurrent": True, "frame": {"duration": surface_play_duration_ms, "redraw": True}, "transition": {"duration": 0}}]},
                {"label": "Pause", "method": "animate", "args": [[None], {"mode": "immediate", "frame": {"duration": 0, "redraw": False}, "transition": {"duration": 0}}]},
            ],
        }
    ],
    sliders=[{"active": 0, "currentvalue": {"prefix": "Source frame "}, "pad": {"t": 35}, "steps": surface_slider_steps}],
)
fig_video_3d.show()


## Capture Output Exact Pixel Lookup

This helper reads the full-resolution raw metric depth file for a processed frame. It requires a capture run created with `--save-frame-depths`.


In [ ]:
def video_depth_at(processed_index, x, y):
    """Return full-resolution metric depth in meters for a processed capture frame and pixel."""
    if not video_has_raw_depths:
        raise ValueError(
            "No raw depth files are available for this capture run. "
            f"Run this first: {capture_rerun_with_depths_command}."
        )
    processed_index = int(processed_index)
    x = int(x)
    y = int(y)
    if not (0 <= processed_index < len(video_depth_files)):
        raise IndexError(f"Processed frame index {processed_index} is outside 0..{len(video_depth_files) - 1}")
    frame_depth = load_depth_array(video_depth_files[processed_index])
    frame_height, frame_width = frame_depth.shape
    if not (0 <= x < frame_width and 0 <= y < frame_height):
        raise IndexError(f"Pixel ({x}, {y}) is outside frame bounds {frame_width}x{frame_height}")
    return float(frame_depth[y, x])

if video_has_raw_depths:
    first_depth = load_depth_array(video_depth_files[0])
    first_center_x = first_depth.shape[1] // 2
    first_center_y = first_depth.shape[0] // 2
    print(
        f"Depth at processed frame 0 / source frame {video_frame_numbers[0]} "
        f"center pixel ({first_center_x}, {first_center_y}): "
        f"{video_depth_at(0, first_center_x, first_center_y):.4f} m"
    )
else:
    print("Raw depth files are not available in this capture run. Use --save-frame-depths to enable exact lookup.")
